# Notebook 01: Data Collection

This notebook collects all raw data for our forensic analysis of Trump's Truth Social posts
and their impact on oil markets during the March 2026 Iran crisis.

**Data sources:**
1. Trump Truth Social posts (scraped from trumpstruth.org)
2. Brent & WTI crude oil daily prices (FRED)
3. DJT stock price (Yahoo Finance via yfinance)
4. Intraday oil proxy data (USO ETF via yfinance)
5. Intraday crude oil data (EODHD API)

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
import json
from urllib.parse import urlparse, parse_qs

# Create output directories
os.makedirs('../data/raw', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Libraries loaded and directories ready.')

Libraries loaded and directories ready.


## 1. Scrape Trump Truth Social Posts

We scrape all posts from January 20, 2025 (inauguration) through March 24, 2026
from trumpstruth.org, a public archive of Trump's Truth Social posts.

The site uses server-rendered HTML with pagination via cursor tokens.

In [3]:
def scrape_truth_posts(start_date, end_date, per_page=100, max_pages=200):
    """
    Scrape Trump's Truth Social posts from trumpstruth.org.
    
    Parameters:
        start_date (str): Start date in YYYY-MM-DD format
        end_date (str): End date in YYYY-MM-DD format
        per_page (int): Posts per page (max 100)
        max_pages (int): Safety limit on pagination
    
    Returns:
        pd.DataFrame with columns: post_id, timestamp_str, post_text, post_url, original_url
    """
    all_posts = []
    base_url = 'https://trumpstruth.org'
    
    # Initial request params
    params = {
        'sort': 'desc',
        'per_page': str(per_page),
        'start_date': start_date,
        'end_date': end_date,
        'removed': 'include'
    }
    
    headers = {'User-Agent': 'Mozilla/5.0 (research project)'}
    page = 0
    
    while page < max_pages:
        page += 1
        
        try:
            resp = requests.get(base_url, params=params, headers=headers, timeout=30)
            resp.raise_for_status()
        except requests.RequestException as e:
            print(f'Request failed on page {page}: {e}')
            break
        
        soup = BeautifulSoup(resp.text, 'html.parser')
        statuses = soup.find_all('div', class_='status')
        
        if not statuses:
            print(f'No posts found on page {page}. Done.')
            break
        
        for status in statuses:
            # Extract post text
            content_div = status.find('div', class_='status__content')
            if content_div:
                paragraphs = content_div.find_all('p')
                post_text = '\n'.join(p.get_text(strip=True) for p in paragraphs)
            else:
                post_text = ''
            
            # Extract timestamp and post URL
            meta_items = status.find_all('a', class_='status-info__meta-item')
            timestamp_str = ''
            post_url = ''
            post_id = ''
            
            for item in meta_items:
                href = item.get('href', '')
                if '/statuses/' in href:
                    timestamp_str = item.get_text(strip=True)
                    post_url = href
                    post_id = href.split('/statuses/')[-1].strip()
            
            # Extract original Truth Social URL
            ext_link = status.find('a', class_='status__external-link')
            original_url = ext_link.get('href', '') if ext_link else ''
            
            # Extract image descriptions if present
            attachments = status.find_all('img', class_='status-attachment__image')
            image_descriptions = [img.get('alt', '') for img in attachments if img.get('alt')]
            
            all_posts.append({
                'post_id': post_id,
                'timestamp_str': timestamp_str,
                'post_text': post_text,
                'post_url': post_url,
                'original_url': original_url,
                'image_descriptions': ' | '.join(image_descriptions) if image_descriptions else ''
            })
        
        print(f'Page {page}: scraped {len(statuses)} posts (total: {len(all_posts)})')
        
        # Find next page link
        pagination = soup.find('div', class_='pagination')
        next_link = None
        if pagination:
            for a in pagination.find_all('a'):
                if 'Next' in a.get_text():
                    next_link = a.get('href')
                    break
        
        if not next_link:
            print('No next page. Scraping complete.')
            break
        
        # Parse cursor from next page URL
        parsed = urlparse(next_link)
        next_params = parse_qs(parsed.query)
        
        if 'cursor' in next_params:
            params['cursor'] = next_params['cursor'][0]
        else:
            print('No cursor in next link. Done.')
            break
        
        # Be polite to the server
        time.sleep(1.5)
    
    df = pd.DataFrame(all_posts)
    print(f'\nTotal posts scraped: {len(df)}')
    return df

In [6]:
# Scrape all posts from inauguration through March 24, 2026
# This may take 10-15 minutes with rate limiting
posts_df = scrape_truth_posts(
    start_date='2025-01-20',
    end_date='2026-03-24',
    per_page=100
)

# Save raw data
posts_df.to_csv('../data/raw/truth_posts.csv', index=False)
print(f'Saved {len(posts_df)} posts to data/raw/truth_posts.csv')
posts_df.head()

Page 1: scraped 102 posts (total: 102)
Page 2: scraped 103 posts (total: 205)
Page 3: scraped 100 posts (total: 305)
Page 4: scraped 102 posts (total: 407)
Page 5: scraped 111 posts (total: 518)
Page 6: scraped 108 posts (total: 626)
Page 7: scraped 107 posts (total: 733)
Page 8: scraped 109 posts (total: 842)
Page 9: scraped 121 posts (total: 963)
Page 10: scraped 142 posts (total: 1105)
Page 11: scraped 107 posts (total: 1212)
Page 12: scraped 140 posts (total: 1352)
Page 13: scraped 107 posts (total: 1459)
Page 14: scraped 147 posts (total: 1606)
Page 15: scraped 132 posts (total: 1738)
Page 16: scraped 101 posts (total: 1839)
Page 17: scraped 109 posts (total: 1948)
Page 18: scraped 147 posts (total: 2095)
Page 19: scraped 110 posts (total: 2205)
Page 20: scraped 141 posts (total: 2346)
Page 21: scraped 132 posts (total: 2478)
Page 22: scraped 104 posts (total: 2582)
Page 23: scraped 107 posts (total: 2689)
Page 24: scraped 145 posts (total: 2834)
Page 25: scraped 131 posts (total:

,post_id,timestamp_str,post_text,post_url,original_url,image_descriptions
0,37467,"March 25, 2026, 2:17 PM",My meeting with the Highly Respected President...,https://trumpstruth.org/statuses/37467,https://truthsocial.com/@realDonaldTrump/11629...,
1,37465,"March 25, 2026, 12:53 PM",We are asking for expedited Judicial Review be...,https://trumpstruth.org/statuses/37465,https://truthsocial.com/@realDonaldTrump/11629...,
2,37466,"March 25, 2026, 12:51 PM",Speaker of the House Mike Johnson and Senate M...,https://trumpstruth.org/statuses/37466,https://truthsocial.com/@realDonaldTrump/11629...,
3,37464,"March 25, 2026, 12:39 PM",I am so proud of our ICE Patriots! They were u...,https://trumpstruth.org/statuses/37464,https://truthsocial.com/@realDonaldTrump/11629...,
4,37463,"March 25, 2026, 11:30 AM","The Radical Left, Country Hating Democrats are...",https://trumpstruth.org/statuses/37463,https://truthsocial.com/@realDonaldTrump/11629...,


In [15]:
# Display the full post_text from a random post
random_post = posts_df.sample(1).iloc[0]
print(random_post['post_text'])



The Creeps at the Failing New York Times are at it again. I won the 2024 Presidential Election in a Landslide, winning all Seven Swing States, the Popular Vote, and the Electoral College by a lot. I one our Nation’s Districts by 2750 to 550, a complete wipeout. I settled 8 Wars, have 48 New Stock Market Highs, our Economy is Great, and our Country is RESPECTED AGAIN all over the World, respected like never before. The last Administration had the Highest Inflation in history - I have already brought that down to normal, and prices, including groceries, are coming down. To do this requires a lot of Work and Energy, and I have never worked so hard in my life. Yet despite all of this the Radical Left Lunatics in the soon to fold New York Times did a hit piece on me that I am perhaps losing my Energy, despite facts that show the exact opposite. They know this is wrong, as is almost every thing that they write about me, including election results, ALL PURPOSELY NEGATIVE. This cheap “RAG” is 

## 2. Download Daily Oil Prices from FRED

FRED (Federal Reserve Economic Data) provides free CSV downloads for:
- **DCOILBRENTEU**: Brent Crude Oil Prices (Europe)
- **DCOILWTICO**: WTI Crude Oil Prices (Cushing, Oklahoma)

Both are in dollars per barrel, updated daily on trading days.

In [7]:
def download_fred_series(series_id, start_date, end_date, save_path):
    """
    Download a FRED time series as CSV.
    
    FRED uses '.' for missing values (weekends/holidays).
    """
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv'
    params = {
        'id': series_id,
        'cosd': start_date,
        'coed': end_date
    }
    
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    
    with open(save_path, 'w') as f:
        f.write(resp.text)
    
    df = pd.read_csv(save_path)
    df.columns = ['date', 'price']
    df['date'] = pd.to_datetime(df['date'])
    df['price'] = pd.to_numeric(df['price'], errors='coerce')  # '.' becomes NaN
    df = df.dropna(subset=['price'])
    
    print(f'{series_id}: {len(df)} trading days, {df["date"].min().date()} to {df["date"].max().date()}')
    print(f'  Price range: ${df["price"].min():.2f} - ${df["price"].max():.2f}')
    
    return df

In [8]:
# Download Brent crude
brent_df = download_fred_series(
    'DCOILBRENTEU', '2025-01-01', '2026-03-24',
    '../data/raw/brent_daily.csv'
)
brent_df = brent_df.rename(columns={'price': 'brent_close'})

# Download WTI crude
wti_df = download_fred_series(
    'DCOILWTICO', '2025-01-01', '2026-03-24',
    '../data/raw/wti_daily.csv'
)
wti_df = wti_df.rename(columns={'price': 'wti_close'})

DCOILBRENTEU: 305 trading days, 2025-01-02 to 2026-03-16
  Price range: $59.93 - $103.23
DCOILWTICO: 298 trading days, 2025-01-02 to 2026-03-16
  Price range: $55.44 - $98.48


## 3. Download DJT Stock Price

Trump Media & Technology Group (DJT) stock price serves as a proxy for
whether Trump personally benefits from market-moving posts.

In [9]:
import yfinance as yf

djt = yf.download('DJT', start='2025-01-01', end='2026-03-25')
djt = djt.reset_index()

# Flatten multi-level columns if present
if isinstance(djt.columns, pd.MultiIndex):
    djt.columns = [col[0] if col[1] == '' or col[1] == 'DJT' else f'{col[0]}_{col[1]}' 
                   for col in djt.columns]

djt = djt.rename(columns={'Date': 'date', 'Close': 'djt_close', 'Volume': 'djt_volume'})
djt['date'] = pd.to_datetime(djt['date']).dt.tz_localize(None)
djt = djt[['date', 'djt_close', 'djt_volume']]

djt.to_csv('../data/raw/djt_stock.csv', index=False)
print(f'DJT: {len(djt)} trading days')
print(f'  Price range: ${djt["djt_close"].min():.2f} - ${djt["djt_close"].max():.2f}')
djt.head()

[*********************100%***********************]  1 of 1 completed

DJT: 306 trading days
  Price range: $8.58 - $42.91


,date,djt_close,djt_volume
0,2025-01-02,34.020000,5166600
1,2025-01-03,34.619999,6150400
2,2025-01-06,36.169998,8425600
3,2025-01-07,35.220001,5504700
4,2025-01-08,34.540001,5525900


## 4. Download Intraday Oil Data

Intraday data is critical for proving **causation**: did the post come BEFORE the price move?

We use the **USO ETF** (United States Oil Fund) via yfinance, which provides 5-minute bars
for the last 60 days. USO closely tracks WTI crude oil futures.

*Note: EODHD's free tier does not support intraday data (403 error), so we rely entirely on yfinance.*

In [14]:
# Key dates for intraday analysis (the most suspicious posts)
KEY_DATES = [
    '2026-03-06',  # "UNCONDITIONAL SURRENDER" post
    '2026-03-09',  # Same-day oscillation ("war complete" -> "hit 20x harder")
    '2026-03-21',  # 48-hour ultimatum
    '2026-03-22',  # "OBLITERATE" post
    '2026-03-23',  # "productive conversations" fabrication
    '2026-03-24',  # 5-day extension
]

# yfinance can get recent intraday data (last 60 days for 5m bars)
# USO ETF closely tracks WTI crude oil
try:
    uso_intraday = yf.download('USO', period='60d', interval='5m')
    uso_intraday = uso_intraday.reset_index()
    
    if isinstance(uso_intraday.columns, pd.MultiIndex):
        uso_intraday.columns = [col[0] if col[1] == '' or col[1] == 'USO' 
                                else f'{col[0]}_{col[1]}' for col in uso_intraday.columns]
    
    uso_intraday = uso_intraday.rename(columns={'Datetime': 'timestamp', 'Close': 'uso_close'})
    uso_intraday.to_csv('../data/raw/uso_intraday.csv', index=False)
    print(f'USO intraday: {len(uso_intraday)} bars')
    print(f'  Date range: {uso_intraday["timestamp"].min()} to {uso_intraday["timestamp"].max()}')
except Exception as e:
    print(f'USO intraday download failed: {e}')
    print('Will fall back to daily-only analysis for causality.')

[*********************100%***********************]  1 of 1 completed

USO intraday: 4491 bars
  Date range: 2025-12-26 14:30:00+00:00 to 2026-03-24 14:25:00+00:00


In [15]:
# EODHD free tier does NOT support intraday data (returns 403).
# The USO ETF intraday data from yfinance (cell above) is our intraday source.
# USO closely tracks WTI crude oil and covers all key March 2026 crisis dates.

import os
if os.path.exists('../data/raw/uso_intraday.csv'):
    uso_check = pd.read_csv('../data/raw/uso_intraday.csv')
    print(f'✅ USO intraday data available: {len(uso_check)} bars')
    print(f'   This will be used for intraday causality analysis in Notebook 05.')
else:
    print('⚠️ No intraday data available. Causality analysis will use daily data only.')

✅ USO intraday data available: 4491 bars
   This will be used for intraday causality analysis in Notebook 05.


## 5. Data Collection Summary

Let's verify what we've collected:

In [16]:
print('=== DATA COLLECTION SUMMARY ===')
print()

# Check what files exist
raw_files = os.listdir('../data/raw')
for f in sorted(raw_files):
    path = f'../data/raw/{f}'
    size = os.path.getsize(path)
    if f.endswith('.csv'):
        df = pd.read_csv(path)
        print(f'{f}: {len(df)} rows, {len(df.columns)} columns ({size/1024:.1f} KB)')
    else:
        print(f'{f}: {size/1024:.1f} KB')

print()
print('Next step: Run notebook 02 to clean and merge these datasets.')

=== DATA COLLECTION SUMMARY ===

brent_daily.csv: 313 rows, 2 columns (5.2 KB)
djt_stock.csv: 306 rows, 3 columns (11.2 KB)
truth_posts.csv: 21014 rows, 6 columns (8900.7 KB)
uso_intraday.csv: 4491 rows, 6 columns (450.7 KB)
wti_daily.csv: 313 rows, 2 columns (5.2 KB)

Next step: Run notebook 02 to clean and merge these datasets.
